## کتابخانه‌های مورد نیاز

- **NumPy (نامپای)** برای عملیات ریاضی
- **Librosa (لایبروزا)** برای کار با فایل صوتی
- **SoundFile (ساندفایل)** برای ذخیره‌سازی فایل صوتی
- **Random (رندوم)** برای تولید جمعیت اولیه
- **OS (سیستم عامل)** برای ساخت فایل‌ها

In [2]:
import numpy as np
import librosa
import soundfile as sf
import random 
import os  


# تنظیمات کانفیگ پروژه

در سلول پایین، کانفیگ‌های پروژه تنظیم شده است.

## پارامترهای اولیه
- **فایل گل** و **پوشه نهایی** در اولین پارامترها نوشته شده است.

## پارامترهای مربوط به MLL
در پارامترهای بعدی به تنظیم پارامترهایی برای استفاده از **مل** پرداخته شده است.  
در این کانفیگ‌ها از معیارهای متداول در دنیای کامپیوتر و تحلیل صوت استفاده شده است.

## مشخصات نمونه‌برداری (Sampling)
- **نرخ نمونه‌برداری (Sample Rate):** `22050`  
  (یعنی در هر ثانیه 22050 نمونه از صدا استخراج می‌کند)
- **مدت زمان هر فایل:** `3` ثانیه

## تبدیل به اسپکتوگرام (تصویر فرکانسی)
برای تبدیل فرکانس صدا به یک اسپکتوگرام:
- **رزولوشن:** `128`
- **طول پنجره برای تبدیل فوریه:** `2048`
- **همپوشانی پنجره‌ها:** `75%`

In [62]:
goalfile = "voice.wav"
output = "generated"
os.makedirs(output, exist_ok=True)
c_sr = 22050
c_duration = 3
c_n_mels= 128
c_n_fft = 2048
c_hop = 512
poplution = 200
gens = 300
eliter = 0.3

در این سلول به لود فایل اصلی و همچنین استخراج اسپکتوگرام پرداخته شده است.

In [63]:
audio,sr=librosa.load(goalfile , sr=c_sr)
print("lenght of voice",len(audio))

audio = audio[:c_sr * c_duration]

audio = audio / np.max(np.abs(audio))
target_mel= librosa.feature.melspectrogram(
    y=audio,
    sr=sr,
    n_fft=c_n_fft,
    hop_length=c_hop,
    n_mels=c_n_mels)
target_mel = librosa.power_to_db(
    target_mel,
    ref=np.max
)
target_shape = target_mel.shape

print("Target shape:", target_shape)

lenght of voice 73030
Target shape: (128, 130)


## توابع الگوریتم ژنتیک

### ساخت نسل اول (Initial Population)
- از کتابخانه `random` استفاده شده است.
- برای دور نشدن از هدف، محدوده را حول داده‌ی اصلی ساخته‌ایم.
- همچنین به تابع `shape` مورد نظر را داده‌ایم.

### تابع Fitness (برازندگی)
- معیار **MSE** (میانگین مربعات خطا) استفاده شده است.
- برای درک بهتر، از **منفی MSE** استفاده کرده‌ایم (هرچه مقدار بالاتر، بهتر).

### تابع Selection (انتخاب)
- با توجه به **الایت ریت (Elite Rate)**، تعدادی از اعضای جامعه که باید انتخاب شوند مشخص می‌گردند.
- بقیه اعضا حذف (drop) می‌شوند.

### تابع Crossover (ترکیب)
- با استفاده از تابع `random` به عمل ترکیب می‌پردازد و فرزند تولید می‌کند.

### تابع Mutation (جهش)
- برای ساخت مقدار محدودی نویز (به منظور جلوگیری از افتادن در بهینه‌ی محلی) استفاده شده است.

### تابع نسل بعدی (Next Generation)
- با استفاده از تابع Crossover فرزند می‌سازد.
- جمعیت جدید به همراه مقدار محدودی نویز ساخته می‌شود.

In [64]:
def create_frist_gen():
    noise = np.random.normal(
        0,
        10,
        target_shape,
    )

    individual = target_mel+ noise

    return individual
    
def create_pup(size):
    population=[]
    for _ in range (size):
        population.append(create_frist_gen())
    return population
    
def fitness_func(individual):

    mse = np.mean(
        (individual - target_mel) ** 2
    )

    return -mse
def select(population , fitnesses):
    sorted_pup= [
        x for _, x in sorted(
            zip(fitnesses, population),
            key=lambda pair: pair[0],
            reverse=True
        )
    ]
    elite_count=int(len(population)* eliter)
    elites= sorted_pup[:elite_count]
    return elites
def crossover(parent1, parent2):

    alpha = np.random.rand()

    child = (
        alpha * parent1
        + (1 - alpha) * parent2
    )

    return child
def mutate(individual, strength):

    individual = individual.copy()

    noise = np.random.normal(
        0,
        strength,
        individual.shape
    )

    individual += noise

    individual = np.clip(
        individual,
        -80,
        20
    )

    return individual
def next_gen(elites,mutation_strength):
    new_pup=elites.copy()
    while len(new_pup)< poplution :
        p1=random.choice(elites)
        p2=random.choice(elites)
        child = crossover(p1,p2)
        child = mutate(child, mutation_strength)
        new_pup.append(child)
    return new_pup

### تابع `revers_file(mel_db)`

این تابع معکوس فرآیند تبدیل صدا به اسپکتوگرام مل را انجام می‌دهد؛ یعنی از یک تصویر فرکانسی (در مقیاس دسی‌بل) دوباره سیگنال صوتی می‌سازد.
### تابع `savef(audio, filename)`

واقعاً ساده‌ترین تابع ممکن:  
با استفاده از کتابخانه `soundfile` آرایه صوتی را با نرخ نمونه‌برداری مشخص (`sr`) در یک فایل ذخیره می‌کند.

In [65]:
def revers_file(mel_db):
    mel_db = np.nan_to_num(
        mel_db,
        nan=-80,
        posinf=20,
        neginf=-80)
    mel_power = librosa.db_to_power(mel_db)
    audio = librosa.feature.inverse.mel_to_audio(
        mel_power,
        sr=c_sr,
        n_fft=c_n_fft,
        hop_length=c_hop)
    audio = np.nan_to_num(audio)
    return audio
def savef(audio,filename):
    sf.write(filename,audio,sr)
    

In [66]:
pup_main = create_pup(poplution)
best_find_score = 0
print("start of ga")
for generation in range(gens):
    mutation_strength = (2*(1 - generation / gens))+ 0.2
    fitnesses = []
    for individual in pup_main:
        score = fitness_func(individual)

        fitnesses.append(score)
    best_inx = np.argmax(fitnesses)
    best_individual = pup_main[best_inx]
    best_fitness = fitnesses[best_inx]
    print(

        f"Generation {generation+1} "

        f"| Best Fitness: "

        f"{best_fitness:.6f}"
    )
    if generation % 10 == 0:
        print("Saving audio...")
        generated_audio = revers_file(
            best_individual
        )
        filename = os.path.join(
            output,
            f"gen_{generation}.wav"
        )
        savef(
            generated_audio,
            filename)
    elite = select(
        pup_main,
        fitnesses)
    elite = elite.copy()
    if generation % 15 == 0:
        for _ in range(5):
            elite.append(
                create_frist_gen())
    pup_main = next_gen(
        elite,
        mutation_strength)
print("\nEvolution finished.")
final_audio = revers_file(best_individual)
savef(final_audio,os.path.join(output,"final_result.wav"))
print("\nSaved final_result.wav")

start of ga
Generation 1 | Best Fitness: -97.009412
Saving audio...
Generation 2 | Best Fitness: -48.221472
Generation 3 | Best Fitness: -29.495110
Generation 4 | Best Fitness: -20.744743
Generation 5 | Best Fitness: -16.398817
Generation 6 | Best Fitness: -14.700697
Generation 7 | Best Fitness: -13.634550
Generation 8 | Best Fitness: -13.634550
Generation 9 | Best Fitness: -13.634550
Generation 10 | Best Fitness: -13.634550
Generation 11 | Best Fitness: -13.634550
Saving audio...
Generation 12 | Best Fitness: -13.634550
Generation 13 | Best Fitness: -13.634550
Generation 14 | Best Fitness: -13.634550
Generation 15 | Best Fitness: -13.634550
Generation 16 | Best Fitness: -13.634550
Generation 17 | Best Fitness: -13.634550
Generation 18 | Best Fitness: -13.546205
Generation 19 | Best Fitness: -13.546205
Generation 20 | Best Fitness: -13.349537
Generation 21 | Best Fitness: -13.349537
Saving audio...
Generation 22 | Best Fitness: -13.349537
Generation 23 | Best Fitness: -13.349537
Genera